# Notebook 03 — Feature Engineering, Encoding, Scaling, and Final Dataset Preparation

This notebook builds on the cleaned and outlier-adjusted LendingClub dataset produced in Notebook 02.
The purpose here is to transform the dataset into a model-ready table using business-driven feature engineering,
transparent encoding of categorical variables, and scaling of continuous features.

The goal is not “fancy ML preprocessing,” but **explainable and auditable PD-model preparation** aligned to
industry standards used in credit risk modelling.

This notebook focuses on:

1. **Financial capacity ratios** – borrower’s ability to repay  
2. **Credit behavior indicators** – how borrowers use and manage credit  
3. **Stress flags** – business-rule thresholds banks use to detect risky customers  
4. **Segmentation buckets** – non-target-based grouping rules for PD models  
5. **Time-based fields** – used only for EDA and out-of-time validation  
6. **Encoding** – transforming categories into numerical form  
7. **Scaling** – standardizing numeric variables for model stability  
8. **Saving final modelling dataset and artifacts**  

Each section includes strong reasoning behind every decision so the notebook stands up to
audit, review, model governance, and recruiter scrutiny.


In [2]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# sklearn StandardScaler (allowed if makes coding/business sense)
from sklearn.preprocessing import StandardScaler
import joblib   # to save scaler, optional
pd.set_option('display.max_columns', None)

In [3]:
in_colab = False
try:
    import google.colab
    in_colab = True
except:
    in_colab = False

if in_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    input_file = "/content/drive/MyDrive/raw/LendingClub/processed/loans_preprocessed_stage2.parquet"
    output_folder = "/content/drive/MyDrive/raw/LendingClub/processed"
else:
    input_file = r"C:\Users\deves\Documents\GitHub\portfolio\pdpit-lendingclub\data\processed\loans_preprocessed_stage2.parquet"
    output_folder = r"C:\Users\deves\Documents\GitHub\portfolio\pdpit-lendingclub\data\processed"
    art_folder = r"C:\Users\deves\Documents\GitHub\portfolio\pdpit-lendingclub\artifacts"

print("Running in Colab:", in_colab)
print("Input:", input_file)
print("Output folder:", output_folder)


Running in Colab: False
Input: C:\Users\deves\Documents\GitHub\portfolio\pdpit-lendingclub\data\processed\loans_preprocessed_stage2.parquet
Output folder: C:\Users\deves\Documents\GitHub\portfolio\pdpit-lendingclub\data\processed


In [4]:
# load file (must exist from Notebook 2)
df = pd.read_parquet(input_file)
print("Loaded dataframe shape:", df.shape)
display(df.head(3))


Loaded dataframe shape: (1794323, 23)


,issue_d,loan_amnt,int_rate,grade,sub_grade,home_ownership,annual_inc,verification_status,purpose,addr_state,dti,delinq_2yrs,inq_last_6mths,open_acc,total_acc,revol_bal,revol_util,loan_status,default_ind,age_of_loan_months,term_months,emp_length_num,emp_length_num_missing
0,2015-12-01,3600.0,13.99,C,C4,MORTGAGE,55000.0,Not Verified,debt_consolidation,PA,5.91,0.0,1.0,7.0,13.0,2765.0,29.7,Fully Paid,0,37,36,10,0
1,2015-12-01,24700.0,11.99,C,C1,MORTGAGE,65000.0,Not Verified,small_business,SD,16.06,1.0,3.0,22.0,38.0,21470.0,19.2,Fully Paid,0,37,36,10,0
2,2015-12-01,20000.0,10.78,B,B4,MORTGAGE,63000.0,Not Verified,home_improvement,IL,10.78,0.0,0.0,6.0,18.0,7869.0,56.2,Fully Paid,0,37,60,10,0


## 2. Financial Capacity & Credit Behavior Ratios

Ratios are the backbone of PD models because they express key financial conditions in normalized form.

### Why ratios matter:
- Borrowers with high loan burden relative to income default more.
- Borrowers with high revolving utilization (revol_bal) show liquidity stress.
- Ratios make features scale-free and comparable across customers.
- Decision makers understand ratios better than raw values.

### Why these specific ratios:
- `income_to_loan_ratio` → repayment capacity  
- `lti` → affordability (industry standard)  
- `utilization_ratio` → credit card stress  
- `open_acc_ratio` → credit utilization structure  
- `loan_to_revol_bal` → over-reliance on revolving lines  
- log transforms smooth heavy right tails  

These features are **not random**: they come from credit underwriting heuristics widely used in retail lending.


In [6]:
df['income_to_loan_ratio'] = df['annual_inc'] / (df['loan_amnt'] + 1)
df['lti'] = df['loan_amnt'] / (df['annual_inc'] + 1)

df['utilization_ratio'] = df['revol_bal'] / (df['loan_amnt'] + 1)
df['loan_to_revol_bal'] = df['loan_amnt'] / (df['revol_bal'] + 1)

df['open_acc_ratio'] = df['open_acc'] / (df['total_acc'] + 1)

# log transforms for skew reduction
df['log_income_to_loan'] = np.log1p(df['income_to_loan_ratio'])
df['log_annual_inc'] = np.log1p(df['annual_inc'])
df['log_revol_bal'] = np.log1p(df['revol_bal'])

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

print("Ratio features created.")
display(df[['lti','income_to_loan_ratio','utilization_ratio','open_acc_ratio']].head())


Ratio features created.


,lti,income_to_loan_ratio,utilization_ratio,open_acc_ratio
0,0.065453,15.273535,0.767842,0.500000
1,0.379994,2.631472,0.869196,0.564103
2,0.317455,3.149843,0.393430,0.315789
3,0.318179,3.142767,0.222908,0.722222
4,0.099584,10.040669,2.108355,0.333333


## 3. Stress Indicators (Business Rules)

These are **not WOE bins**.  
They are **business-rule flags** used by lenders to quickly identify high-risk borrowers.

### Why these rules:
- `dti > 50` → borrower devotes over half income to debt repayments  
- `utilization_ratio > 0.75` → borrower is close to maxing out credit  
- `inq_last_6mths >= 3` → active credit-seeking behavior  
- `loan > 50% of income` → affordability warning  
- Combined stress (high utilization + high DTI) is extremely predictive  

These rules are **explainable**, **interpretable**, **audit-friendly**, and provide non-linear signal to the model.


In [8]:
df['high_dti_flag'] = (df['dti'] > 50).astype("Int64")
df['high_util_flag'] = (df['utilization_ratio'] > 0.75).astype("Int64")
df['recent_inq_flag'] = (df['inq_last_6mths'] >= 1).astype("Int64")
df['high_inq_flag'] = (df['inq_last_6mths'] >= 3).astype("Int64")
df['loan_gt_half_income_flag'] = (df['loan_amnt'] > 0.5 * df['annual_inc']).astype("Int64")

df['high_util_and_dti'] = (df['high_util_flag'] * df['high_dti_flag']).astype("Int64")

print("Stress indicators created.")


Stress indicators created.


## 4. Segmentation Buckets (Not WOE)

These buckets divide borrowers into **interpretable groups** used in credit analysis.  
This is **not** target-driven binning (that comes later in scorecard development).

Buckets help PD models capture non-linear risk effects without overcomplicating transformations.

We create:

### Employment stability buckets  
- 0 years  
- '1–3 years'  
- '4–6 years'  
- '7–10 years'  

### Term buckets  
- 36 months  
- 60 months  

### Loan age buckets  
- '0–6 months'  
- '7–12 months'  
- '13–24 months'  
- '25+ months'  

Each bucket corresponds to clear business intuition about repayment behavior.


In [10]:
df.columns

Index(['issue_d', 'loan_amnt', 'int_rate', 'grade', 'sub_grade',
       'home_ownership', 'annual_inc', 'verification_status', 'purpose',
       'addr_state', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc',
       'total_acc', 'revol_bal', 'revol_util', 'loan_status', 'default_ind',
       'age_of_loan_months', 'term_months', 'emp_length_num',
       'emp_length_num_missing', 'income_to_loan_ratio', 'lti',
       'utilization_ratio', 'loan_to_revol_bal', 'open_acc_ratio',
       'log_income_to_loan', 'log_annual_inc', 'log_revol_bal',
       'high_dti_flag', 'high_util_flag', 'recent_inq_flag', 'high_inq_flag',
       'loan_gt_half_income_flag', 'high_util_and_dti'],
      dtype='object')

In [11]:
df['emp_len_0'] = (df['emp_length_num'] == 0).astype("Int64")
df['emp_len_1_3'] = ((df['emp_length_num'] >= 1) & (df['emp_length_num'] <= 3)).astype("Int64")
df['emp_len_4_6'] = ((df['emp_length_num'] >= 4) & (df['emp_length_num'] <= 6)).astype("Int64")
df['emp_len_7_10'] = ((df['emp_length_num'] >= 7) & (df['emp_length_num'] <= 10)).astype("Int64")

df['term_36'] = (df['term_months'] == 36).astype("Int64")
df['term_60'] = (df['term_months'] == 60).astype("Int64")

df['age_0_6m'] = (df['age_of_loan_months'] <= 6).astype("Int64")
df['age_7_12m'] = ((df['age_of_loan_months'] > 6) & (df['age_of_loan_months'] <= 12)).astype("Int64")
df['age_13_24m'] = ((df['age_of_loan_months'] > 12) & (df['age_of_loan_months'] <= 24)).astype("Int64")
df['age_25_plus_m'] = (df['age_of_loan_months'] > 24).astype("Int64")

print("Segmentation buckets created.")


Segmentation buckets created.


## 5. Time-Based Features (For EDA & Stability Only)

These features help:
- Perform out-of-time validation  
- Study stability across vintages  
- Detect shifts in borrower behaviour  

They MUST NOT be used as modelling inputs, because time is not a borrower attribute.


In [13]:
df['issue_year'] = df['issue_d'].dt.year
df['issue_quarter'] = df['issue_d'].dt.to_period('Q').astype(str)

print("Time-based fields created for EDA and stability (not for modeling).")


Time-based fields created for EDA and stability (not for modeling).


## 6. Encoding Categorical Variables

We use a simple, transparent rule:

- If a category has **≤ 6 unique values** → **One-Hot Encoding**  
- If it has **> 6 unique values** → **Frequency Encoding**

### Why this rule?
- One-hot preserves interpretability for small controlled categories  
- Frequency encoding prevents explosion of columns  
- Both approaches are easy to audit and reproduce  
- WOE encoding will be done later during scorecard creation  


In [15]:
cat_cols = []
for c in df.columns:
    dt = str(df[c].dtype).lower()
    if ('object' in dt) or ('category' in dt):
        cat_cols.append(c)

print("Categorical columns:", cat_cols)


Categorical columns: ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'loan_status', 'issue_quarter']


In [16]:
# Cell 15 - Decide which categorical columns use one-hot vs frequency encoding
threshold = 10   # you requested threshold = 10

one_hot_cols = []
freq_cols = []
for c in cat_cols:
    nuniq = df[c].nunique(dropna=True)
    print(c, "unique =", nuniq)
    if nuniq <= threshold:
        one_hot_cols.append(c)
    else:
        freq_cols.append(c)

print("One-hot (<= {} uniques):".format(threshold), one_hot_cols)
print("Frequency encode (>{} uniques):".format(threshold), freq_cols)


grade unique = 7
sub_grade unique = 35
home_ownership unique = 5
verification_status unique = 3
purpose unique = 14
addr_state unique = 50
loan_status unique = 7
issue_quarter unique = 16
One-hot (<= 10 uniques): ['grade', 'home_ownership', 'verification_status', 'loan_status']
Frequency encode (>10 uniques): ['sub_grade', 'purpose', 'addr_state', 'issue_quarter']


#### “In this notebook, all feature transformations (ratios, flags, one-hot encoding, frequency encoding, scaling) are unsupervised and do not use the target. Therefore, they do not cause target leakage. For PIT PD development, I perform the dataset split in Notebook 05 just before model training, because supervised transformations such as WOE will be done after the split.”

In [18]:
# Cell 16 - One-hot encode columns with <= 10 uniques using sklearn OneHotEncoder

from sklearn.preprocessing import OneHotEncoder
import joblib
import os

# Make OneHotEncoder compatible with both old and new sklearn versions
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')


print("Fitting OneHotEncoder on:", one_hot_cols)

# Fit encoder
encoder.fit(df[one_hot_cols].astype(str))

# Transform
ohe_arr = encoder.transform(df[one_hot_cols].astype(str))
ohe_feature_names = encoder.get_feature_names_out(one_hot_cols)

# Convert to DataFrame
ohe_df = pd.DataFrame(ohe_arr, columns=ohe_feature_names, index=df.index)

# Cast to Int64 for readability
for col in ohe_df.columns:
    ohe_df[col] = ohe_df[col].astype("Int64")

# Add to df
df = pd.concat([df, ohe_df], axis=1)
print("Added", len(ohe_feature_names), "one-hot columns")

# Save encoder
onehot_file = os.path.join(output_folder, "artifacts", "onehot_encoder.pkl")
os.makedirs(os.path.dirname(onehot_file), exist_ok=True)
joblib.dump(encoder, onehot_file)
print("Saved OneHotEncoder to:", onehot_file)


Fitting OneHotEncoder on: ['grade', 'home_ownership', 'verification_status', 'loan_status']
Added 22 one-hot columns
Saved OneHotEncoder to: C:\Users\deves\Documents\GitHub\portfolio\pdpit-lendingclub\data\processed\artifacts\onehot_encoder.pkl


In [19]:
# Cell 17 - Frequency encode columns with > threshold uniques and save mapping

import os

freq_map_rows = []
total = len(df)

print("Frequency-encoding columns:", freq_cols)

for c in freq_cols:
    print(c)
    counts = df[c].value_counts(dropna=False)
    new_col = c + "_freq"
    df[new_col] = df[c].map(counts) / total
    print("  created:", new_col, "(unique values:", counts.shape[0], ")")
    for value, cnt in counts.items():
        freq_map_rows.append((c, str(value), int(cnt), float(cnt / total)))

# Save freq mapping CSV
freq_map_file = os.path.join(output_folder, "artifacts", "freq_encodings.csv")
os.makedirs(os.path.dirname(freq_map_file), exist_ok=True)
fm = pd.DataFrame(freq_map_rows, columns=['column','value','count','freq'])
fm.to_csv(freq_map_file, index=False)
print("Saved frequency mapping to:", freq_map_file)


Frequency-encoding columns: ['sub_grade', 'purpose', 'addr_state', 'issue_quarter']
sub_grade
  created: sub_grade_freq (unique values: 35 )
purpose
  created: purpose_freq (unique values: 14 )
addr_state
  created: addr_state_freq (unique values: 50 )
issue_quarter
  created: issue_quarter_freq (unique values: 16 )
Saved frequency mapping to: C:\Users\deves\Documents\GitHub\portfolio\pdpit-lendingclub\data\processed\artifacts\freq_encodings.csv


In [20]:
# Cell 18 - show a small sample of created features for inspection

created_ohe = list(ohe_feature_names) if 'ohe_feature_names' in globals() else []
created_freq = [c + "_freq" for c in freq_cols]

# Pick up to 8 of each to display
ohe_show = created_ohe[:8]
freq_show = created_freq[:8]

print("One-hot columns sample:", ohe_show)
print("Freq-encoded columns sample:", freq_show)

cols_to_display = []
cols_to_display += ohe_show
cols_to_display += freq_show
cols_to_display = [c for c in cols_to_display if c in df.columns]

if len(cols_to_display) > 0:
    display(df[cols_to_display].head(6))
else:
    print("No encoded columns found to display (nothing to show).")


One-hot columns sample: ['grade_A', 'grade_B', 'grade_C', 'grade_D', 'grade_E', 'grade_F', 'grade_G', 'home_ownership_ANY']
Freq-encoded columns sample: ['sub_grade_freq', 'purpose_freq', 'addr_state_freq', 'issue_quarter_freq']


,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,home_ownership_ANY,sub_grade_freq,purpose_freq,addr_state_freq,issue_quarter_freq
0,0,0,1,0,0,0,0,0,0.057407,0.559344,0.033726,0.072732
1,0,0,1,0,0,0,0,0,0.066289,0.009851,0.001989,0.072732
2,0,1,0,0,0,0,0,0,0.060906,0.069060,0.040439,0.072732
3,0,0,1,0,0,0,0,0,0.052692,0.559344,0.036264,0.072732
4,0,0,0,0,0,1,0,0,0.005279,0.022636,0.033726,0.072732
5,0,0,1,0,0,0,0,0,0.057888,0.559344,0.033004,0.072732


In [21]:
# Cell 19 - select only continuous numeric variables for scaling

scale_cols = [
    'loan_amnt',
    'int_rate',
    'annual_inc',
    'dti',
    'delinq_2yrs',
    'inq_last_6mths',
    'open_acc',
    'total_acc',
    'revol_bal',
    'revol_util',
    'age_of_loan_months'
]

print("Columns selected for scaling:", len(scale_cols))
print(scale_cols)


Columns selected for scaling: 11
['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'total_acc', 'revol_bal', 'revol_util', 'age_of_loan_months']


In [22]:
scaler = StandardScaler()

df_scale = df[scale_cols].fillna(0.0)
scaler.fit(df_scale.values)
scaled = scaler.transform(df_scale.values)

for i, c in enumerate(scale_cols):
    df[c + "_sc"] = scaled[:, i]

scaler_file = os.path.join(art_folder, "scaler.pkl")
joblib.dump(scaler, scaler_file)

print("Scaling completed for continuous columns.")


Scaling completed for continuous columns.


In [23]:
# Cell 21 - Drop original continuous variables now that scaled versions exist

to_drop = [
    'loan_amnt',
    'int_rate',
    'annual_inc',
    'dti',
    'delinq_2yrs',
    'inq_last_6mths',
    'open_acc',
    'total_acc',
    'revol_bal',
    'revol_util',
    'age_of_loan_months'
]

df = df.drop(columns=to_drop)

print("Dropped original continuous features:")
print(to_drop)


Dropped original continuous features:
['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'total_acc', 'revol_bal', 'revol_util', 'age_of_loan_months']


In [24]:
# Cell 22 - Quick sanity check of scaled columns

scaled_cols = [c + "_sc" for c in scale_cols]

print("Scaled columns describe (first 10 shown):")
display(df[scaled_cols[:10]].describe())


Scaled columns describe (first 10 shown):


,loan_amnt_sc,int_rate_sc,annual_inc_sc,dti_sc,delinq_2yrs_sc,inq_last_6mths_sc,open_acc_sc,total_acc_sc,revol_bal_sc,revol_util_sc
count,1.794323e+06,1.794323e+06,1.794323e+06,1.794323e+06,1.794323e+06,1.794323e+06,1.794323e+06,1.794323e+06,1.794323e+06,1.794323e+06
mean,8.927309e-17,5.265386e-16,3.352176e-16,-7.361664e-16,-9.635744e-17,-5.608080e-17,-3.887086e-17,9.404879e-17,-3.447373e-16,4.095973e-16
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-1.514260e+00,-1.559007e+00,-1.680830e+00,-1.832175e+00,-4.042976e-01,-6.553096e-01,-2.094398e+00,-1.861624e+00,-9.748289e-01,-1.975929e+00
25%,-7.697135e-01,-7.103551e-01,-6.585174e-01,-6.435272e-01,-4.042976e-01,-6.553096e-01,-6.586396e-01,-7.535777e-01,-6.204339e-01,-7.695337e-01
50%,-2.378943e-01,-1.212422e-01,-2.669933e-01,-9.064675e-02,-4.042976e-01,-6.553096e-01,-1.202303e-01,-1.569372e-01,-2.936929e-01,-2.059692e-02
75%,5.066525e-01,5.626231e-01,3.855467e-01,5.341902e-01,-4.042976e-01,6.346276e-01,5.976488e-01,6.101721e-01,2.541576e-01,7.607263e-01
max,2.633929e+00,3.009707e+00,4.409544e+00,1.647573e+01,5.136631e+00,3.214502e+00,3.289695e+00,3.167203e+00,5.186143e+00,2.072378e+00


In [25]:
# Cell 23 - Save final transformed dataset

output_folder = r"C:\Users\deves\Documents\GitHub\portfolio\pdpit-lendingclub\data\processed"
os.makedirs(output_folder, exist_ok=True)

final_file = os.path.join(output_folder, "loans_transformed_for_modeling.parquet")
df.to_parquet(final_file, index=False)

sample_file = os.path.join(art_folder, "loans_transformed_sample.csv")
df.sample(min(len(df), 5000), random_state=42).to_csv(sample_file, index=False)

print("Saved transformed dataset:")
print(final_file)

print("\nSaved sample:")
print(sample_file)


Saved transformed dataset:
C:\Users\deves\Documents\GitHub\portfolio\pdpit-lendingclub\data\processed\loans_transformed_for_modeling.parquet

Saved sample:
C:\Users\deves\Documents\GitHub\portfolio\pdpit-lendingclub\artifacts\loans_transformed_sample.csv


In [26]:
# Cell 24 - Final summary

print("Notebook 03 completed.")
print("One-hot encoded columns added:", len([c for c in df.columns if '_' in c and any(col in c for col in ['grade_', 'home_ownership_', 'verification_status_', 'loan_status_'])]))
print("Frequency encoded columns:", len([c for c in df.columns if c.endswith('_freq')]))
print("Scaled continuous columns:", len([c for c in df.columns if c.endswith('_sc')]))
print("")
print("Artifacts saved:")
print("- scaler.pkl")
print("- onehot_encoder.pkl")
print("- freq_encodings.csv")
print("- loans_transformed_sample.csv")
print("")
print("Next: Notebook 04 (EDA, correlations, stability, feature selection).")


Notebook 03 completed.
One-hot encoded columns added: 23
Frequency encoded columns: 4
Scaled continuous columns: 11

Artifacts saved:
- scaler.pkl
- onehot_encoder.pkl
- freq_encodings.csv
- loans_transformed_sample.csv

Next: Notebook 04 (EDA, correlations, stability, feature selection).
